## How to generate the best command to draft metabolic models from MAGs with CarveMe-GutMicrobes 

### Load bin taxonomy and reference taxonomy tables

We start by importing the bin classification table (`bins_taxonomy.tsv`) and a reference
taxonomy table (`taxonomy_agora.tsv`). These will be used to map each bin to a validated
taxonomic level and name.

In [1]:
import pandas as pd

# Load bin-level classifications
df = pd.read_csv("bins_taxonomy.tsv", sep="\t")

# Load reference taxonomy (Agora database)
agora = pd.read_csv("taxonomy_agora.tsv", sep="\t")

### Define rank mapping and hierarchical order

We define a mapping from the standard prefixes in CarveMe-GutMicrobes taxonomies
(s__, g__, f__, etc.) to full rank names. We also specify the hierarchy
from most specific (species) to most general (kingdom), to allow
fallback to higher ranks if needed.

In [2]:
# Map CarveMe prefixes to taxonomic ranks
rank_map = {
    "s__":"species",
    "g__":"genus",
    "f__": "family",
    "o__": "order",
    "c__": "class",
    "p__": "phylum",
    "d__": "kingdom"
}

# Hierarchical order from most specific to most general
rank_order = ["species","genus","family","order", "class", "phylum", "kingdom"]

### Build sets of valid taxonomic names from the reference

For each rank, we extract the unique names from the reference taxonomy.
These sets will be used to check whether a bin's taxonomic assignment
is valid according to the reference database.

In [3]:
# Create sets of valid names for each rank
agora_sets = {
    "species": set(agora["Species"].dropna().str.strip()),
    "genus": set(agora["Genus"].dropna().str.strip()),
    "family": set(agora["Family"].dropna().str.strip()),
    "order": set(agora["Order"].dropna().str.strip()),
    "class": set(agora["Class"].dropna().str.strip()),
    "phylum": set(agora["Phylum"].dropna().str.strip()),
    "kingdom": set(agora["Kingdom"].dropna().str.strip()),
}

### Parse each MAGs taxonomy and select the most specific valid taxonomic level

For each MAG, we:
1. Split its classification into levels (species, genus, etc.).
2. Try to select the most specific level with a non-empty name.
3. If no valid level is found, we skip the MAG

In [4]:
for _, row in df.iterrows():
    bin_id = row["bin"]
    tax = row["classification"]

    levels = tax.split(";")

    chosen_level = None
    chosen_name = None

    # First attempt: select the most specific non-empty level
    for level in reversed(levels):
        if "__" in level:
            prefix, name = level.split("__", 1)
            if name.strip() != "":
                key = prefix + "__"
                if key in rank_map:
                    chosen_level = rank_map[key]
                    chosen_name = name.strip()
                    break

    if chosen_level is None:
        print(f"# skipped {bin_id} (no taxonomy)")
        continue

### Validate the chosen taxonomic assignment

We check whether the most specific taxonomic level selected for each MAG exists
in the reference taxonomy incorporated (AGORA2). If it does, we will use it; if not,
we may need to fallback to higher ranks. MAGs without any valid assignment
are skipped.

In [5]:
# Get index of the current rank in the hierarchical order
current_rank_index = rank_order.index(chosen_level)

final_level = None
final_name = None

### Fallback to higher ranks if needed

If the chosen name is not present in the reference, we iteratively check
higher (less specific) ranks according to the hierarchy. We pick the first
rank with a valid name present in the reference. This ensures that each MAG
gets a taxonomic annotation supported by the reference database.

In [6]:
for i in range(current_rank_index, len(rank_order)):
    rank = rank_order[i]
    name = chosen_name if i == current_rank_index else None

    # If moving up in the hierarchy, get the name from the original classification
    if i != current_rank_index:
        for level in reversed(levels):
            if level.startswith(rank[0] + "__"):
                name = level.split("__", 1)[1].strip()
                break

    # Use this rank if the name exists in the reference
    if name and name in agora_sets[rank]:
        final_level = rank
        final_name = name
        break

    # Skip bin if no valid assignment was found
    if final_level is None:
        print(f"# skipped {bin_id} (not found in taxonomy_agora)")
    continue
    

# skipped CRR029092_bin.9 (not found in taxonomy_agora)


### Generate CarveMe-GutMicrobes command for the MAG

Once a valid taxonomic level and name are assigned, we construct the
CarveMe-GutMicrobes command to build a genome-scale metabolic model (GEM) for the MAG.
The command specifies:
- the input FASTA file
- the taxonomic level and name
- output file
- FBC2 format

In [7]:
for _, row in df.iterrows():
    bin_id = row["bin"]
    tax = row["classification"]

    levels = tax.split(";")

    chosen_level = None
    chosen_name = None

    # primo tentativo: livello più specifico disponibile
    for level in reversed(levels):
        if "__" in level:
            prefix, name = level.split("__", 1)
            if name.strip() != "":
                key = prefix + "__"
                if key in rank_map:
                    chosen_level = rank_map[key]
                    chosen_name = name.strip()
                    break

    if chosen_level is None:
        print(f"# skipped {bin_id} (no taxonomy)")
        continue

    # controllo contro taxonomy_agora e fallback gerarchico
    current_rank_index = rank_order.index(chosen_level)

    final_level = None
    final_name = None

    for i in range(current_rank_index, len(rank_order)):
        rank = rank_order[i]
        name = chosen_name if i == current_rank_index else None

        # se saliamo di livello, prendiamo il nome dal taxonomy originale
        if i != current_rank_index:
            for level in reversed(levels):
                if level.startswith(rank[0] + "__"):
                    name = level.split("__", 1)[1].strip()
                    break

        if name and name in agora_sets[rank]:
            final_level = rank
            final_name = name
            break

    if final_level is None:
        print(f"# skipped {bin_id} (not found in taxonomy_agora)")
        continue

    fasta = f"{bin_id}.faa"
    cmd = f'carvemegut-carve {fasta} -tl {final_level} -t "{final_name}" --fbc2 -o {bin_id}.xml.gz'
    print(cmd)

carvemegut-carve CRR029083_bin.1.faa -tl genus -t "Acetatifactor" --fbc2 -o CRR029083_bin.1.xml.gz
carvemegut-carve CRR029083_bin.10.faa -tl genus -t "Anaerotignum" --fbc2 -o CRR029083_bin.10.xml.gz
carvemegut-carve CRR029083_bin.12.faa -tl genus -t "Prevotella" --fbc2 -o CRR029083_bin.12.xml.gz
carvemegut-carve CRR029083_bin.17.faa -tl genus -t "Gemmiger" --fbc2 -o CRR029083_bin.17.xml.gz
carvemegut-carve CRR029083_bin.18.faa -tl class -t "Clostridia" --fbc2 -o CRR029083_bin.18.xml.gz
carvemegut-carve CRR029083_bin.19.faa -tl genus -t "Mediterraneibacter" --fbc2 -o CRR029083_bin.19.xml.gz
carvemegut-carve CRR029083_bin.21.faa -tl genus -t "Agathobaculum" --fbc2 -o CRR029083_bin.21.xml.gz
carvemegut-carve CRR029083_bin.22.faa -tl family -t "Pasteurellaceae" --fbc2 -o CRR029083_bin.22.xml.gz
carvemegut-carve CRR029083_bin.23.faa -tl species -t "Parabacteroides distasonis" --fbc2 -o CRR029083_bin.23.xml.gz
carvemegut-carve CRR029083_bin.24.faa -tl family -t "Lachnospiraceae" --fbc2 -o CR

THE END :) 